# VAQ Evaluation Plots

This notebook mirrors the visual style of `lsqpp_relerr_plots.ipynb`, but evaluates **VAQ** results from:

`/data/cpanourg/2-hdvc/results/vaq/vaq_bpv_grid_min2-4-6_max8-16-32_bpv4-8-12_20260501_104000`

Plots generated:

- Avg relative error vs `bits_per_vector`
- Avg relative error vs `min_bits`
- Avg relative error vs `max_bits`
- Avg relative error vs training time
- ADC time vs `bits_per_vector`
- ADC time vs `min_bits`
- ADC time vs `max_bits`
- Avg relative error vs ADC time with Pareto frontier
- Companion versions of the parameter plots show one remaining hyperparameter in the legend.

Rows with missing relative error are excluded from relative-error plots only; timing plots still use the available timing rows.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# --- Plot style: intentionally aligned with lsqpp_relerr_plots.ipynb ---
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 40,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 21,
})
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

PROJECT_ROOT = Path("/home/cpanourg/projects/2-hdvc")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path("/data/cpanourg/2-hdvc/results/vaq/vaq_bpv_grid_FIXED_20260503_150802")
FIGURES_DIR = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METHODS_TO_PLOT = ["VAQ"]
DATASETS_TO_PLOT = ["deep", "bigann", "gist", "msmarco", "openai"]
FIGURE_SAVE_FORMATS = ("pdf", "svg")

CONFIG_COLS = ["method", "dataset", "bits_per_vector", "min_bits", "max_bits", "variance"]
ADC_TIME_COL = "adc_time_per_pair_s"
ADC_TIME_LABEL = "ADC time"


In [ ]:
# Color and marker palettes copied from the LSQ++ notebook.
COLOR_PALETTE = [
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red",
    "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan",
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red"
]

MARKER_PALETTE = [
    "o", "v", "s", "^", "D", "<", ">", "p", "*", "h", "H", "X", "d", "P", "8"
]

def create_dynamic_color_map(unique_values):
    sorted_values = sorted(unique_values)
    return {val: COLOR_PALETTE[i % len(COLOR_PALETTE)] for i, val in enumerate(sorted_values)}


def save_figure(fig, output_dir: Path, stem: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for ext in FIGURE_SAVE_FORMATS:
        path = output_dir / f"{stem}.{ext}"
        fig.savefig(path, bbox_inches="tight")
        paths.append(path)
    print("Saved " + " and ".join(str(p) for p in paths))


def style_axes(ax, tick_fontsize=40, grid_axis="y"):
    ax.grid(True, axis=grid_axis, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=tick_fontsize)


def adjust_ylabel_position(ax, y_label: str):
    if y_label == "Avg Relative Error":
        ax.yaxis.set_label_coords(-0.13, 0.39)


def set_sci_axes(ax):
    for axis in [ax.xaxis, ax.yaxis]:
        fmt = ScalarFormatter(useMathText=True)
        fmt.set_powerlimits((-2, 3))
        axis.set_major_formatter(fmt)


In [ ]:
def load_vaq_data(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    csv_paths = sorted(data_dir.glob("*_VAQ_adc_vs_exact_eval.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No VAQ CSV files found in {data_dir}")

    frames = []
    for path in csv_paths:
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = path.name.split("_")[0]
        if "method" not in df.columns:
            df["method"] = "VAQ"
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    numeric_cols = [
        "bits_per_vector", "min_bits", "max_bits", "variance",
        "train_size", "adc_time_s", "rel_error_mean", "rel_error_std",
        "train_time_s", "encoding_time_s", "distance_table_time_s",
        "cdist_time_s", "nb_sample", "nq_sample", "dim", "nb", "nq",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["method"] = df["method"].replace({"VAQ": "VAQ"})
    df["pair_count"] = df["nb_sample"] * df["nq_sample"]
    df[ADC_TIME_COL] = df["adc_time_s"] / df["pair_count"]
    df["bpv_ratio"] = df["bits_per_vector"] / df["dim"]

    required = ["method", "dataset", "bits_per_vector", "min_bits", "max_bits", "variance", "adc_time_s"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return df

raw_df = load_vaq_data(DATA_DIR)
print("Rows per dataset:")
display(raw_df.groupby("dataset").size())
raw_df.head()


In [ ]:
def aggregate_metric_df(df: pd.DataFrame, y_cols=("rel_error_mean", ADC_TIME_COL)) -> pd.DataFrame:
    agg_spec = {c: "mean" for c in y_cols if c in df.columns}
    for c in [
        "rel_error_std", "distance_table_time_s", "cdist_time_s",
        "train_time_s", "encoding_time_s", "dim", "nb", "nb_sample",
        "nq_sample", "pair_count", "adc_time_s", "bpv_ratio",
    ]:
        if c in df.columns and c not in agg_spec:
            agg_spec[c] = "mean" if pd.api.types.is_numeric_dtype(df[c]) else "first"
    return (
        df.groupby(CONFIG_COLS, as_index=False)
        .agg(agg_spec)
        .sort_values(["dataset", "bits_per_vector", "min_bits", "max_bits"])
    )

plot_df = aggregate_metric_df(raw_df)
plot_df.groupby("dataset").size()


In [ ]:
def plot_avg_metric_vs_param(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    y_col: str,
    y_label: str,
    output_stem: str,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
    x_log: bool = False,
    xtick_count: int | None = None,
):
    """Average y over all other hyperparameters, then plot one figure per dataset/method."""
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(subset=[x_col, y_col]).copy()
            if sub.empty:
                continue
            agg = sub.groupby([x_col], as_index=False)[y_col].agg(["mean", "std"]).reset_index().sort_values(x_col)
            if agg.empty:
                continue

            fig, ax = plt.subplots()
            ax.errorbar(
                agg[x_col], agg["mean"], yerr=agg["std"].fillna(0),
                fmt="o-", color=COLOR_PALETTE[0], markersize=16,
                linewidth=2.5, markeredgewidth=2, markeredgecolor="black",
                capsize=5, capthick=2, elinewidth=1.5,
            )
            if x_log:
                ax.set_xscale("log")
            ax.set_xlabel(x_label, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            adjust_ylabel_position(ax, y_label)
            style_axes(ax, tick_fontsize=40, grid_axis="y")
            if x_col in ("bits_per_vector", "min_bits", "max_bits", "bpv_ratio"):
                vals = sorted(agg[x_col].dropna().unique())
                if xtick_count is not None and len(vals) > xtick_count:
                    tick_idx = np.linspace(0, len(vals) - 1, xtick_count, dtype=int)
                    vals = [vals[i] for i in sorted(set(tick_idx))]
                ax.set_xticks(vals)
                ax.set_xticklabels([str(int(v)) if float(v).is_integer() else f"{v:g}" for v in vals], rotation=0)
            set_sci_axes(ax)
            plt.tight_layout()
            stem = f"{output_stem}_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)


In [ ]:
def _format_legend_value(col: str, value):
    if pd.isna(value):
        return f"{col}=nan"
    if col == "min_bits":
        return f"min={int(value)}"
    if col == "max_bits":
        return f"max={int(value)}"
    if col == "bits_per_vector":
        return f"bpv={int(value)}"
    if col == "bpv_ratio":
        return f"bpv/d={value:g}"
    if isinstance(value, (int, np.integer)) or (isinstance(value, float) and value.is_integer()):
        return f"{col}={int(value)}"
    return f"{col}={value}"


def _legend_label(row, legend_cols):
    return ", ".join(_format_legend_value(col, row[col]) for col in legend_cols)


def plot_metric_vs_param_with_config_legend(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    y_col: str,
    y_label: str,
    legend_cols: list,
    output_stem: str,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
    x_log: bool = False,
):
    """Plot y vs one parameter, drawing one curve for each setting of one other hyperparameter."""
    for method in methods:
        for dataset in datasets:
            needed = [x_col, y_col] + legend_cols
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(subset=needed).copy()
            if sub.empty:
                continue

            agg = (
                sub.groupby([x_col] + legend_cols, as_index=False)[y_col]
                .mean()
                .sort_values(legend_cols + [x_col])
            )
            if agg.empty:
                continue

            legend_settings = agg[legend_cols].drop_duplicates().sort_values(legend_cols).reset_index(drop=True)
            fig, ax = plt.subplots()
            for idx, setting in legend_settings.iterrows():
                mask = np.ones(len(agg), dtype=bool)
                for col in legend_cols:
                    mask &= agg[col].eq(setting[col]).to_numpy()
                curve = agg.loc[mask].sort_values(x_col)
                if curve.empty:
                    continue
                color = COLOR_PALETTE[idx % len(COLOR_PALETTE)]
                marker = MARKER_PALETTE[idx % len(MARKER_PALETTE)]
                ax.plot(
                    curve[x_col], curve[y_col], marker=marker, linestyle="-",
                    color=color, markersize=14, linewidth=2.5,
                    markeredgewidth=1.5, markeredgecolor="black",
                    label=_legend_label(setting, legend_cols),
                )

            if x_log:
                ax.set_xscale("log")
            ax.set_xlabel(x_label, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            adjust_ylabel_position(ax, y_label)
            style_axes(ax, tick_fontsize=40, grid_axis="y")
            if x_col in ("bits_per_vector", "min_bits", "max_bits", "bpv_ratio"):
                vals = sorted(agg[x_col].dropna().unique())
                ax.set_xticks(vals)
                ax.set_xticklabels([str(int(v)) if float(v).is_integer() else f"{v:g}" for v in vals], rotation=0)
            set_sci_axes(ax)
            ax.legend(frameon=True, fontsize=16, loc="best")
            plt.tight_layout()
            stem = f"{output_stem}_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)


## Avg Relative Error Plots

In [ ]:
# Averaged trends
plot_avg_metric_vs_param(
    plot_df,
    x_col="bits_per_vector",
    x_label="Bits per vector",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    output_stem="avg_relerr_vs_bits_per_vector",
    xtick_count=4,
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="min_bits",
    x_label="min bits",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    output_stem="avg_relerr_vs_min_bits",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="max_bits",
    x_label="max bits",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    output_stem="avg_relerr_vs_max_bits",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="train_time_s",
    x_label="Training time (s)",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    output_stem="avg_relerr_vs_train_time",
)

# Companion plots: one hyperparameter per legend
plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col="bits_per_vector",
    x_label="Bits per vector",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    legend_cols=["min_bits"],
    output_stem="avg_relerr_vs_bits_per_vector_legend_min_bits",
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col="bits_per_vector",
    x_label="Bits per vector",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    legend_cols=["max_bits"],
    output_stem="avg_relerr_vs_bits_per_vector_legend_max_bits",
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col="min_bits",
    x_label="min bits",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    legend_cols=["max_bits"],
    output_stem="avg_relerr_vs_min_bits_legend_max_bits",
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col="max_bits",
    x_label="max bits",
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    legend_cols=["min_bits"],
    output_stem="avg_relerr_vs_max_bits_legend_min_bits",
)


## ADC Time Plots

In [ ]:
plot_avg_metric_vs_param(
    plot_df,
    x_col="bits_per_vector",
    x_label="Bits per vector",
    y_col=ADC_TIME_COL,
    y_label=ADC_TIME_LABEL,
    output_stem="adc_time_vs_bits_per_vector",
    xtick_count=4,
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="min_bits",
    x_label="min bits",
    y_col=ADC_TIME_COL,
    y_label=ADC_TIME_LABEL,
    output_stem="adc_time_vs_min_bits",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="max_bits",
    x_label="max bits",
    y_col=ADC_TIME_COL,
    y_label=ADC_TIME_LABEL,
    output_stem="adc_time_vs_max_bits",
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col="min_bits",
    x_label="min bits",
    y_col=ADC_TIME_COL,
    y_label=ADC_TIME_LABEL,
    legend_cols=["max_bits"],
    output_stem="adc_time_vs_min_bits_legend_max_bits",
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col="max_bits",
    x_label="max bits",
    y_col=ADC_TIME_COL,
    y_label=ADC_TIME_LABEL,
    legend_cols=["min_bits"],
    output_stem="adc_time_vs_max_bits_legend_min_bits",
)


In [ ]:
def pareto_frontier_minimize(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    """Return nondominated lower-left points: lower x and lower y are better."""
    pts = df.sort_values([x_col, y_col]).copy()
    frontier_rows = []
    best_y = np.inf
    for _, row in pts.iterrows():
        if row[y_col] < best_y:
            frontier_rows.append(row)
            best_y = row[y_col]
    if not frontier_rows:
        return pts.iloc[0:0]
    return pd.DataFrame(frontier_rows)


def plot_pareto_adc_vs_metric(
    df: pd.DataFrame,
    y_col: str,
    y_label: str,
    output_stem: str,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
):
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(subset=[ADC_TIME_COL, y_col]).copy()
            if sub.empty:
                continue

            agg = (
                sub.groupby(["bits_per_vector", "min_bits", "max_bits"], as_index=False)
                .agg({ADC_TIME_COL: "mean", y_col: "mean"})
                .sort_values([ADC_TIME_COL, y_col])
            )
            frontier = pareto_frontier_minimize(agg, ADC_TIME_COL, y_col)

            fig, ax = plt.subplots()
            ax.scatter(
                agg[ADC_TIME_COL], agg[y_col],
                color="tab:blue", marker="o", s=260,
                edgecolors="black", linewidths=2, alpha=0.75,
            )
            if len(frontier) > 0:
                ax.plot(frontier[ADC_TIME_COL], frontier[y_col], "-", color="gray", linewidth=1.5, alpha=0.8, zorder=1)
                ax.scatter(
                    frontier[ADC_TIME_COL], frontier[y_col],
                    color="#D32F2F", marker="o", s=330,
                    edgecolors="black", linewidths=2, zorder=3,
                )

            label_offsets = [(10, 10), (12, -18), (-62, 12), (-64, -20), (16, 24), (-72, 26)]
            for idx, (_, row) in enumerate(frontier.iterrows()):
                label = f"({int(row['bits_per_vector'])}, {int(row['min_bits'])}, {int(row['max_bits'])})"
                offset = label_offsets[idx % len(label_offsets)]
                ax.annotate(
                    label,
                    (row[ADC_TIME_COL], row[y_col]),
                    xytext=offset, textcoords="offset points",
                    fontsize=13, color="black", alpha=0.95,
                    bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.72),
                    arrowprops=dict(arrowstyle="-", color="0.35", lw=0.8, shrinkA=0, shrinkB=5),
                    zorder=4,
                )

            ax.set_xlabel(ADC_TIME_LABEL, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            adjust_ylabel_position(ax, y_label)
            style_axes(ax, tick_fontsize=40, grid_axis="both")
            set_sci_axes(ax)
            plt.tight_layout()
            stem = f"{output_stem}_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)

plot_pareto_adc_vs_metric(
    plot_df,
    y_col="rel_error_mean",
    y_label="Avg Relative Error",
    output_stem="pareto_relerr_vs_adc_time",
)


In [ ]:

def plot_best_worst_relerr_vs_bpv(
    df: pd.DataFrame,
    dataset: str = "deep",
    method: str = "VAQ",
    output_dir: Path = FIGURES_DIR,
):
    """Show attainable best/worst relative error over hyperparameter choices at each BPV."""
    sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(
        subset=["bits_per_vector", "min_bits", "max_bits", "rel_error_mean"]
    ).copy()
    if sub.empty:
        raise ValueError(f"No rows for {method}/{dataset}")

    best = sub.loc[sub.groupby("bits_per_vector")["rel_error_mean"].idxmin()].sort_values("bits_per_vector")
    worst = sub.loc[sub.groupby("bits_per_vector")["rel_error_mean"].idxmax()].sort_values("bits_per_vector")

    fig, ax = plt.subplots(figsize=(11, 6))
    rng = np.random.default_rng(123)
    for bpv, grp in sub.groupby("bits_per_vector"):
        jitter = rng.uniform(-0.018, 0.018, size=len(grp)) * sub["bits_per_vector"].max()
        ax.scatter(
            grp["bits_per_vector"] + jitter,
            grp["rel_error_mean"],
            s=160,
            color="0.72",
            edgecolor="black",
            linewidth=1.2,
            alpha=0.8,
            zorder=2,
        )

    ax.plot(best["bits_per_vector"], best["rel_error_mean"], "o-", color="tab:green", linewidth=3,
            markersize=14, markeredgecolor="black", label="Best config", zorder=4)
    ax.plot(worst["bits_per_vector"], worst["rel_error_mean"], "o-", color="tab:red", linewidth=3,
            markersize=14, markeredgecolor="black", label="Worst config", zorder=4)

    for label_df, color, yoff in [(best, "tab:green", -22), (worst, "tab:red", 14)]:
        for _, row in label_df.iterrows():
            label = f"min={int(row['min_bits'])}, max={int(row['max_bits'])}"
            ax.annotate(
                label,
                (row["bits_per_vector"], row["rel_error_mean"]),
                xytext=(8, yoff), textcoords="offset points",
                fontsize=12, color=color,
                bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.78),
                arrowprops=dict(arrowstyle="-", color="0.4", lw=0.8, shrinkA=0, shrinkB=5),
                zorder=5,
            )

    ax.set_xlabel("Bits per vector", fontsize=40)
    ax.set_ylabel("Avg Relative Error", fontsize=40)
    adjust_ylabel_position(ax, "Avg Relative Error")
    vals = sorted(sub["bits_per_vector"].dropna().unique())
    ax.set_xticks(vals)
    ax.set_xticklabels([str(int(v)) for v in vals])
    style_axes(ax, tick_fontsize=34, grid_axis="y")
    set_sci_axes(ax)
    ax.legend(frameon=True, fontsize=18, loc="best")
    plt.tight_layout()
    stem = f"best_worst_relerr_vs_bits_per_vector_{method.lower()}_{dataset}"
    save_figure(fig, output_dir, stem)
    png = output_dir / f"{stem}.png"
    fig.savefig(png, dpi=180, bbox_inches="tight")
    print(f"Saved {png}")
    plt.show()
    plt.close(fig)

    summary = pd.concat(
        [best.assign(bound="best"), worst.assign(bound="worst")],
        ignore_index=True,
    )[["bound", "bits_per_vector", "min_bits", "max_bits", "rel_error_mean", "rel_error_std", ADC_TIME_COL]]
    summary_path = output_dir / f"best_worst_relerr_vs_bits_per_vector_{method.lower()}_{dataset}.csv"
    summary.to_csv(summary_path, index=False)
    print(f"Saved {summary_path}")
    return summary

best_worst_deep = plot_best_worst_relerr_vs_bpv(plot_df, dataset="deep", method="VAQ")
display(best_worst_deep)
